# An Image is Worth 16x16 Words (ViT) — Implementation / 구현

**Paper**: Dosovitskiy, A., Beyer, L., Kolesnikov, A., Weissenborn, D., Zhai, X., Unterthiner, T., Dehghani, M., Minderer, M., Heigold, G., Gelly, S., Uszkoreit, J., & Houlsby, N. (2021). *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale.* ICLR 2021. arXiv:2010.11929.

## Goals / 목표

**English** — Implement a minimal Vision Transformer (ViT) in PyTorch and train it on a small CIFAR-10 subset. The pieces:
1. `PatchEmbedding` (Conv2d with kernel=stride=patch)
2. Learnable `[class]` token and 1D learnable positional embedding
3. Transformer encoder (pre-LN MSA + MLP)
4. Classification MLP head
5. Train on CIFAR-10 (subset) and visualize attention from the [class] token to image patches

**한국어** — 최소 ViT를 PyTorch로 구현하고 CIFAR-10 부분집합에서 학습합니다. 구성: (1) `PatchEmbedding` — `Conv2d(kernel=stride=patch)`, (2) 학습 가능 `[class]` 토큰 + 1D learnable 위치 임베딩, (3) pre-LN Transformer encoder (MSA + MLP), (4) 분류 MLP head. 학습 후 [class] 토큰이 패치들에 주는 attention을 시각화합니다.

**Kernel** — `study-with-ai`.

In [ ]:
"""Imports and reproducibility setup."""
import math
import random
from typing import Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")
print(f"PyTorch: {torch.__version__}")

## Part 1: Patch Embedding / 패치 임베딩

**English** — Equation (1) of the paper turns an image $x \in \mathbb{R}^{H\times W\times C}$ into a sequence of $N = HW/P^2$ patch tokens of dimension $D$. The classic implementation reshapes the image into patches and applies a linear layer; the equivalent (and faster) form is `nn.Conv2d(C, D, kernel_size=P, stride=P)`. With $H{=}W{=}32$ and $P{=}4$ (CIFAR-10), we get $N = 64$ patches per image.

**한국어** — 논문 식 (1)은 이미지 $x \in \mathbb{R}^{H\times W\times C}$를 $N = HW/P^2$개의 $D$-차원 패치 토큰 시퀀스로 변환합니다. 고전적 구현은 reshape + linear, 동등한 빠른 구현은 `nn.Conv2d(C, D, kernel_size=P, stride=P)`. CIFAR-10($H{=}W{=}32$)에 $P{=}4$로 나누면 $N = 64$ 패치.

In [ ]:
class PatchEmbedding(nn.Module):
    """Splits an image into non-overlapping P x P patches and projects each to D dims.

    Implements Eq. (1) of Dosovitskiy et al. (2021). The Conv2d with kernel=stride=patch_size
    is mathematically equivalent to flattening each patch and applying a single linear layer.

    Args:
        img_size: Side length H = W of the input image (assumed square).
        patch_size: Side length P of each patch.
        in_channels: Number of input channels C.
        embed_dim: Output token dimension D.
    """

    def __init__(self, img_size: int = 32, patch_size: int = 4, in_channels: int = 3, embed_dim: int = 192):
        super().__init__()
        assert img_size % patch_size == 0, "img_size must be divisible by patch_size"
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2  # N
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Returns tokens of shape (B, N, D)."""
        # x: (B, C, H, W) -> proj: (B, D, H/P, W/P)
        x = self.proj(x)
        # Flatten spatial dims and transpose to (B, N, D)
        x = x.flatten(2).transpose(1, 2)
        return x


# Sanity check / 검증: 32x32 image, 4x4 patches -> 64 tokens
_pe = PatchEmbedding(img_size=32, patch_size=4, in_channels=3, embed_dim=192)
_dummy = torch.randn(2, 3, 32, 32)
_out = _pe(_dummy)
print(f"PatchEmbedding output shape: {tuple(_out.shape)}  (expected (2, 64, 192))")

## Part 2: Transformer Encoder Block / Transformer 인코더 블록

**English** — Each encoder block (Eqs. 2-3) is pre-LayerNorm followed by Multi-Head Self-Attention with a residual, then pre-LayerNorm followed by a 2-layer MLP (4× hidden, GELU) with another residual. We expose attention weights so we can visualize them later.

**한국어** — 각 인코더 블록(식 2-3)은 pre-LN → MSA → residual, 그 다음 pre-LN → MLP(4× hidden, GELU) → residual. 시각화를 위해 attention weights를 반환할 수 있도록 노출합니다.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """Vanilla multi-head self-attention with optional attention-weight return."""

    def __init__(self, embed_dim: int, num_heads: int, attn_dropout: float = 0.0, proj_dropout: float = 0.0):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=True)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(attn_dropout)
        self.proj_drop = nn.Dropout(proj_dropout)

    def forward(self, x: torch.Tensor, return_attn: bool = False) -> Tuple[torch.Tensor, torch.Tensor]:
        """Runs MSA on tokens of shape (B, N, D)."""
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # each: (B, h, N, d_h)
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, h, N, N)
        attn = attn.softmax(dim=-1)
        attn_drop = self.attn_drop(attn)
        out = (attn_drop @ v).transpose(1, 2).reshape(B, N, D)
        out = self.proj_drop(self.proj(out))
        return (out, attn) if return_attn else (out, None)


class TransformerEncoderBlock(nn.Module):
    """Pre-LN Transformer encoder block: LN -> MSA + residual -> LN -> MLP + residual."""

    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, attn_dropout=dropout, proj_dropout=dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, return_attn: bool = False) -> Tuple[torch.Tensor, torch.Tensor]:
        attn_out, attn_weights = self.attn(self.norm1(x), return_attn=return_attn)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x, attn_weights


# Sanity check / 검증
_blk = TransformerEncoderBlock(embed_dim=192, num_heads=6, mlp_ratio=4.0, dropout=0.0)
_z = torch.randn(2, 65, 192)  # 64 patches + 1 [class] token
_out, _attn = _blk(_z, return_attn=True)
print(f"Block output: {tuple(_out.shape)}  attention: {tuple(_attn.shape)}")

## Part 3: Full Vision Transformer / 완전한 ViT

**English** — Compose the pieces into the full ViT. Input is fed through `PatchEmbedding`, a learnable `[class]` token is prepended, a learnable 1D positional embedding $E_{\text{pos}}$ is added, $L$ Transformer encoder blocks run sequentially, then the final `[class]` token state is normalized and sent through a one-hidden-layer MLP head (per the paper's pre-training head). Total parameter count for the small config below should be around 2-3M — far smaller than ViT-Base (86M) but with the exact same architecture.

**한국어** — 전체 ViT를 조립합니다. 입력 → `PatchEmbedding` → 학습 가능 `[class]` 토큰 prepend → 1D 학습 가능 위치 임베딩 $E_{\text{pos}}$ 더하기 → $L$개 Transformer encoder block → 마지막 `[class]` 토큰을 LayerNorm → MLP head(논문의 pre-training head 따름). 아래 작은 config 기준 파라미터 약 2-3M (ViT-Base 86M의 1/30). 구조는 동일.

In [ ]:
class VisionTransformer(nn.Module):
    """Minimal Vision Transformer (Dosovitskiy et al., 2021).

    Args:
        img_size: Input image side length.
        patch_size: Patch side length P.
        in_channels: Number of image channels C.
        num_classes: Number of output classes K.
        embed_dim: Token dimension D.
        depth: Number of Transformer encoder blocks L.
        num_heads: Number of attention heads h.
        mlp_ratio: MLP hidden expansion factor (paper: 4.0).
        dropout: Dropout probability inside blocks.
    """

    def __init__(
        self,
        img_size: int = 32,
        patch_size: int = 4,
        in_channels: int = 3,
        num_classes: int = 10,
        embed_dim: int = 192,
        depth: int = 6,
        num_heads: int = 6,
        mlp_ratio: float = 4.0,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        # Pre-training head: MLP with one hidden layer + GELU. (Fine-tuning head would be a single Linear.)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Tanh(),
            nn.Linear(embed_dim, num_classes),
        )
        self._init_weights()

    def _init_weights(self) -> None:
        """Initialize positional embedding, [class] token, and Linear/LayerNorm modules."""
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.zeros_(m.bias)
                nn.init.ones_(m.weight)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        """Runs ViT forward; optionally returns attention from the last block."""
        B = x.shape[0]
        x = self.patch_embed(x)  # (B, N, D)
        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, D)
        x = torch.cat([cls, x], dim=1)  # (B, N+1, D)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        last_attn = None
        for i, blk in enumerate(self.blocks):
            need_attn = return_attn and (i == len(self.blocks) - 1)
            x, attn = blk(x, return_attn=need_attn)
            if need_attn:
                last_attn = attn
        x = self.norm(x)
        cls_repr = x[:, 0]
        logits = self.head(cls_repr)
        return (logits, last_attn) if return_attn else logits


# Build a small ViT for CIFAR-10 and report parameter count.
model = VisionTransformer(
    img_size=32, patch_size=4, in_channels=3, num_classes=10,
    embed_dim=192, depth=6, num_heads=6, mlp_ratio=4.0, dropout=0.1,
).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"Sequence length (incl. [class]): {model.patch_embed.num_patches + 1}")

## Part 4: Train on a CIFAR-10 Subset / CIFAR-10 부분집합 학습

**English** — We train the small ViT on a CIFAR-10 subset (5000 train / 1000 test) for a few epochs. This is a tiny, illustrative run — the paper trains on JFT-300M for weeks of TPU compute. The point here is correctness, not SOTA. Adam + cosine LR schedule.

**한국어** — CIFAR-10 부분집합(학습 5000, 테스트 1000)에서 작은 ViT를 몇 에폭 학습합니다. 이는 정확성 검증용 미니 실험 — 논문은 JFT-300M에서 수 주의 TPU 컴퓨트를 사용. Adam + cosine LR schedule.

In [ ]:
BATCH_SIZE = 128
EPOCHS = 8
LR = 3e-4
WEIGHT_DECAY = 0.05
TRAIN_SUBSET = 5000
TEST_SUBSET = 1000

train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
test_tf = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_full = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_tf)
test_full = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)

rng = np.random.default_rng(SEED)
train_idx = rng.choice(len(train_full), TRAIN_SUBSET, replace=False)
test_idx = rng.choice(len(test_full), TEST_SUBSET, replace=False)
train_set = Subset(train_full, train_idx.tolist())
test_set = Subset(test_full, test_idx.tolist())

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))

def evaluate(net: nn.Module, loader: DataLoader) -> Tuple[float, float]:
    """Returns (loss, accuracy) on the given loader."""
    net.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = net(x)
            loss_sum += criterion(logits, y).item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += x.size(0)
    return loss_sum / total, correct / total

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
for epoch in range(1, EPOCHS + 1):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        scheduler.step()
        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += x.size(0)
    train_loss, train_acc = loss_sum / total, correct / total
    test_loss, test_acc = evaluate(model, test_loader)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f} acc {train_acc:.3f} | test loss {test_loss:.4f} acc {test_acc:.3f}")

## Part 5: Training Curves / 학습 곡선

**English** — Quick sanity-check plots: loss and accuracy over epochs. With only 5000 training images, ViT will not match a CNN — that is exactly the paper's point about needing scale. The aim is just to confirm the architecture trains.

**한국어** — loss와 accuracy 추이. 5000 이미지로 ViT는 CNN을 못 따라잡을 것 — 이것이 바로 논문의 'scale이 필요하다'는 핵심 메시지를 보여주는 일례. 여기서는 구조가 학습되는지만 확인.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_axis = list(range(1, len(history["train_loss"]) + 1))
axes[0].plot(epochs_axis, history["train_loss"], marker="o", label="train")
axes[0].plot(epochs_axis, history["test_loss"], marker="s", label="test")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross-entropy loss")
axes[0].set_title("Loss / 손실")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(epochs_axis, history["train_acc"], marker="o", label="train")
axes[1].plot(epochs_axis, history["test_acc"], marker="s", label="test")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_title("Accuracy / 정확도")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 6: Attention Visualization / Attention 시각화

**English** — One of ViT's most striking properties (Figure 6 of the paper) is that the [class] token's attention to image patches highlights semantically meaningful regions. We extract attention from the *final* encoder block, take the row corresponding to [class] (index 0), drop the [class]-to-[class] entry, average over heads, reshape into the patch grid, and overlay it on the input image.

**한국어** — ViT의 가장 인상적인 성질 중 하나(논문 Figure 6) — [class] 토큰의 패치들에 대한 attention이 의미 있는 영역을 강조합니다. 마지막 인코더 블록에서 attention을 추출, [class] 행(인덱스 0)만 가져와 self-loop를 빼고, head 평균 후 패치 그리드로 reshape하여 입력 이미지에 overlay.

In [ ]:
def cls_attention_map(net: VisionTransformer, image_norm: torch.Tensor) -> np.ndarray:
    """Compute the [class]-to-patches attention map averaged over heads, reshaped to a 2D grid.

    Args:
        net: Trained ViT model.
        image_norm: A single normalized image of shape (1, C, H, W).
    Returns:
        np.ndarray of shape (G, G) where G = H/P (the patch grid side).
    """
    net.eval()
    with torch.no_grad():
        _, attn = net(image_norm.to(device), return_attn=True)
    # attn: (1, h, N+1, N+1). Take row 0 (CLS query), drop column 0 (CLS key), average over heads.
    cls_to_patches = attn[0, :, 0, 1:].mean(dim=0).cpu().numpy()  # (N,)
    grid = int(math.sqrt(cls_to_patches.shape[0]))
    return cls_to_patches.reshape(grid, grid)

def denormalize(image_norm: torch.Tensor) -> np.ndarray:
    """Undo CIFAR-10 normalization for plotting; returns HWC float in [0, 1]."""
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
    img = image_norm.cpu().squeeze(0) * std + mean
    return img.permute(1, 2, 0).clamp(0.0, 1.0).numpy()

CLASS_NAMES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")

n_show = 6
fig, axes = plt.subplots(2, n_show, figsize=(2.4 * n_show, 5))
for col in range(n_show):
    img, label = test_set[col]
    img_b = img.unsqueeze(0)
    with torch.no_grad():
        pred = net_pred = model(img_b.to(device)).argmax(dim=1).item()
    attn_map = cls_attention_map(model, img_b)
    attn_resized = np.kron(attn_map, np.ones((4, 4)))  # upsample 4x via Kronecker product to 32x32
    raw = denormalize(img_b)
    axes[0, col].imshow(raw)
    axes[0, col].set_title(f"true: {CLASS_NAMES[label]}\npred: {CLASS_NAMES[pred]}", fontsize=9)
    axes[0, col].axis("off")
    axes[1, col].imshow(raw)
    axes[1, col].imshow(attn_resized, cmap="jet", alpha=0.45)
    axes[1, col].set_title("[class] -> patch attention", fontsize=9)
    axes[1, col].axis("off")
plt.tight_layout()
plt.show()

print("Top row: input image. Bottom row: input + averaged attention from [class] token to each patch.")
print("On a tiny 5k-image train set the maps are noisy, but you can already see the [class] token")
print("focusing on object-bearing regions rather than background.")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Patch tokenization / 패치 토큰화 | $x_p^i \in \mathbb{R}^{P^2 \cdot C}$, projected by $E$; equivalent to `Conv2d(C, D, P, P)` | `timm.PatchEmbed`, Swin's hierarchical patch merging, MAE's random patch sampling |
| [class] token / [class] 토큰 | BERT-style learnable token whose post-encoder state is the image representation | Often replaced by global average pooling (DeiT showed both are competitive) |
| Position embedding / 위치 임베딩 | 1D learnable; learns 2D structure on its own | Swin's relative position bias, RoPE, ALiBi, conditional positional encodings |
| Encoder block / 인코더 블록 | Pre-LN MSA + 4× MLP with GELU | Same blueprint reused in DeiT, Swin, BEiT, MAE, DINO, CLIP, SAM |
| Pre-training / 사전학습 | Supervised classification on JFT-300M (303M images, 18k classes) | Self-supervised (MAE, BEiT, DINOv2), vision-language (CLIP, SigLIP) |
| Inductive bias / 귀납적 편향 | Almost none — model learns 2D from data | Hybrid models (ConvNeXt, MaxViT) reintroduce locality at scale |

**English** — This minimal implementation reproduces the architectural backbone of ViT exactly: `Conv2d`-based patch embedding, learnable [class] token, learnable 1D positional embedding, pre-LN Transformer encoder, MLP head. With only 5000 training images we cannot reproduce the headline 88.55% — the paper's central thesis is that ViT *needs* JFT-300M-scale pre-training to surpass CNNs. The attention visualization in Part 6 mirrors Figure 6 of the paper at a tiny scale: the [class] token already focuses on object regions rather than background.

**한국어** — 이 최소 구현은 ViT의 구조적 backbone을 그대로 재현합니다: `Conv2d` 기반 패치 임베딩, 학습 가능 [class] 토큰, 학습 가능 1D 위치 임베딩, pre-LN Transformer encoder, MLP head. 5000 이미지만으로는 88.55%를 재현할 수 없습니다 — 논문의 핵심 주장이 바로 'ViT는 JFT-300M 규모 사전학습이 *필요하다*'는 것입니다. Part 6의 attention 시각화는 작은 규모로나마 논문 Figure 6를 모방 — [class] 토큰이 배경이 아닌 객체 영역에 집중하는 모습이 이미 관찰됩니다.